### Script to Run Forward Opt.

What to do first:
* Create a production directory (if it doesn't exist already) and place "insert_file_name_here.ipynb", "wrf_settings.py", and "make_namelist.py".
* In that directory create an "em_fwd_opt" directory.
* In em_adj, the following files are required:
    * GENPARM.TBL
    * LANDUSE.TBL
    * RRTMG_LW_DATA
    * RRTMG_SW_DATA
    * RRTM_DATA
    * VEGPARM.TBL
    * wrf.exe
    * plus.io_config
    * wrfbdy_d01






In [1]:
import numpy as np
import netCDF4
from netCDF4 import Dataset
from wrf import getvar
import os
import subprocess
from subprocess import PIPE
import sys
import glob
import importlib
import pandas as pd
import xarray as xr

sys.path.append(os.path.abspath('..'))

import wrf_settings
import make_namelist

In [3]:
#Select experiment to load settings for

EXP_NAME = 'WRF_Florence_test'

importlib.reload(wrf_settings)

settings = wrf_settings.get_settings(EXP_NAME)
WRF_DIR           = settings['WRF_dir']
BOX_SIZE          = settings['box_size']
ADJ_JC            = settings['adj_jc']
ADJ_IC            = settings['adj_ic']

RUN_HOURS         = settings['run_hours']
START_YEAR          = settings['start_year']
START_MONTH         = settings['start_month']
START_DAY           = settings['start_day']
START_HOUR          = settings['start_hour']
END_YEAR            = settings['end_year']
END_MONTH           = settings['end_month']
END_DAY             = settings['end_day']
END_HOUR            = settings['end_hour']
E_WE               = settings['e_we']
E_SN               = settings['e_sn']
DX                 = settings['dx']
DY                 = settings['dy']
TIME_STEP          = settings['time_step']
INTERVAL_SECONDS   = settings['interval_seconds']
INTERVAL_SECONDS_ADJ = settings['interval_seconds_adj']
DRESPONSE_VALUE = settings['dresponse_value']


print(settings)



{'WRF_dir': '/Users/ngordillo/florence/', 'adj_jc': 119, 'adj_ic': 181, 'box_size': 10, 'run_hours': '36', 'start_year': '2018', 'start_month': '09', 'start_day': '09', 'start_hour': '00', 'end_year': '2018', 'end_month': '09', 'end_day': '10', 'end_hour': '12', 'interval_seconds': '3600', 'interval_seconds_adj': '10800', 'time_step': '60', 'e_we': 350, 'e_sn': 240, 'dx': 18000, 'dy': 18000, 'dresponse_value': -150}


In [4]:
# Create experiment directory and subdirectories for namelists and WRF data (if not created already)

os.chdir(WRF_DIR)
command_mkdir = "mkdir exp_files"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

os.chdir(WRF_DIR + "exp_files/")
command_mkdir = "mkdir " + EXP_NAME
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

os.chdir(WRF_DIR + "exp_files/" + EXP_NAME)      

command_mkdir = "mkdir namelists"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

command_mkdir = "mkdir wrf_data"
output_mkdir=subprocess.run(command_mkdir,shell=True, stdout=PIPE)

mkdir: cannot create directory ‘exp_files’: File exists
mkdir: cannot create directory ‘WRF_Florence_test’: File exists
mkdir: cannot create directory ‘namelists’: File exists
mkdir: cannot create directory ‘wrf_data’: File exists


### Section 1: Preturb for Forward Opt

In [ ]:
# Copy wrfinput_d01 to em_fwd_opt 

os.chdir(WRF_DIR + 'em_fwd_opt/')
command_cp = "cp /Users/brookezibton/s4_folders/s4_data/WRF_DATA/FLORENCE_201809090000/NCEP_18km/wrf_folder/wrfinput_d01 ."
output_trj=subprocess.run(command_cp,shell=True, stdout=PIPE)


In [ ]:
# Load the perturbations from the NetCDF file

perts = xr.open_dataset(WRF_DIR + "exp_files/" + EXP_NAME + '/wrf_data/perturbations.nc')

du = perts['du'].values
dv = perts['dv'].values
dT = perts['dT'].values
dq = perts['dq'].values

perts.close()

In [ ]:
#Add` the perturbations to the initial conditions for the fwd_opt run

new_IC_filename = WRF_DIR + 'em_fwd_opt/wrfinput_d01'  # Your filename
perturbed_IC = Dataset(new_IC_filename, 'r+')  # Dataset is the class behavior to open the file

u0 = perturbed_IC.variables['U']
v0 = perturbed_IC.variables['V']
T0 = perturbed_IC.variables['T']
q0 = perturbed_IC.variables['QVAPOR']


utotal = np.asarray(u0) + np.asarray(du)
vtotal = np.asarray(v0) + np.asarray(dv)
Ttotal = np.asarray(T0) + np.asarray(dT)
Qtotal = np.asarray(q0) + np.asarray(dq)

u0[:,:,:,:] = utotal
v0[:,:,:,:] = vtotal
T0[:,:,:,:] = Ttotal
q0[:,:,:,:] = Qtotal

perturbed_IC.close()

In [ ]:
# dR=DRESPONSE_VALUE


# new_IC_filename = WRF_DIR + 'em_fwd_opt/wrfinput_d01'  # Your filename
# perturbed_IC = Dataset(new_IC_filename, 'r+')  # Dataset is the class behavior to open the file
# u0 = perturbed_IC.variables['U']

# print(dR)

# dR_check=np.sum(uin*du) + np.sum(vin*dv) + np.sum(tin*dT)+ np.sum(qin*dq)
# print('Expected change: ',dR_check, 'compared with specified change: ', dR)

### Section 2: Run FWD_Opt


In [15]:
# make dirs for namelist, and outputsif they don't exist and wrfinputs if they don't exist

output_file = os.path.join(WRF_DIR, "exp_files/" + EXP_NAME + "/namelists/namelist.input.fwd." + EXP_NAME) 

proceed = True

#Comment out the following block if you want to overwrite existing namelist without warning.

if os.path.exists(output_file):
    response = input(f"Warning: A namelist already exists at {output_file}. Overwrite it? (y/n): ")
    
    if response.lower() not in ['y', 'yes']:
        print("Skipping namelist generation.")
        proceed = False 
        
####

if(proceed):

    make_namelist.generate_namelist(
        run_hours = RUN_HOURS,
        start_year = START_YEAR,
        start_month = START_MONTH,
        start_day = START_DAY,
        start_hour = START_HOUR,
        end_year = END_YEAR,
        end_month = END_MONTH,
        end_day = END_DAY,
        end_hour = END_HOUR,
        time_step = TIME_STEP,
        interval_seconds = INTERVAL_SECONDS,
        e_we = E_WE,
        e_sn = E_SN,
        dx = DX,
        dy = DY,    
        wrf_dir = WRF_DIR,
        exp_name = EXP_NAME,

        run_type="fwd"

    )


Success! 'namelist.input.fwd.WRF_Florence_test' has been generated.


In [16]:

#Link FWD namelist to namelist.input

os.chdir(WRF_DIR + 'em_fwd_opt/')
command_linktrj = "ln -sf " + WRF_DIR + "exp_files/" + EXP_NAME + "_namelist.input.fwd " + WRF_DIR + "em_tlm/namelist.input"
output_linktrj=subprocess.run(command_linktrj,shell=True, stdout=PIPE)

In [ ]:
command_runwrf = "source ~/.bashrc && mpirun -np 40 ./wrf.exe"
output_runwrf = subprocess.run(command_runwrf, shell=True, executable='/bin/bash', stdout=subprocess.PIPE)